# Load Cleaned Data

In [42]:
import sys
sys.path.append("..")
from src.data.data_cleaning import split_and_clean_pipeline

results = []

train_df, val_df, test_df, params = split_and_clean_pipeline()

In [43]:
train_df.head(5)


,id,difficulty_rating,strategy_importance,injury_risk_comfort,individual_brilliance,play_location,play_season,associated_music,associated_foods_drinks,associated_color,...,emotional_response_tfidf_like,emotional_response_tfidf_nervous,emotional_response_tfidf_neutral,emotional_response_tfidf_relaxed,emotional_response_tfidf_sport,emotional_response_tfidf_team,emotional_response_tfidf_tense,emotional_response_tfidf_time,emotional_response_tfidf_watch,emotional_response_tfidf_watching
0,1,4.0,Agree,Agree,Strongly disagree,"Indoor facility,Outdoor public space,School or...","Winter,Spring,Summer,Fall","Hip-hop or rap,Latin,Folk or traditional","Beer, chips, skewers",White,...,0.0,0.0,0.0,0.0,0.0,0.0,1.0,0.0,0.0,0.0
1,1,6.0,Neutral,Disagree,Agree,"School or university facility,Private sports f...","Winter,Spring,Summer,Fall",Hip-hop or rap,"Hot dog, soda",Blue,...,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0
2,1,7.0,Strongly agree,Neutral,Strongly agree,"Indoor facility,School or university facility,...","Winter,Spring,Summer,Fall","Pop,Rock,Hip-hop or rap","Soda, popcorn",Orange,...,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0
3,2,4.0,Agree,Agree,Strongly agree,"Indoor facility,Outdoor public space,School or...","Spring,Summer","Pop,Electronic or dance,Latin,Folk or traditional","French fries, soda, beer",green,...,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0
4,2,8.0,Neutral,Agree,Strongly agree,"Outdoor public space,School or university faci...","Spring,Summer,Fall","Rock,Folk or traditional","Hot dogs, pizza",Blue,...,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0


In [44]:
train_df.columns
print(train_df.columns.tolist())

['id', 'difficulty_rating', 'strategy_importance', 'injury_risk_comfort', 'individual_brilliance', 'play_location', 'play_season', 'associated_music', 'associated_foods_drinks', 'associated_color', 'associated_song', 'emotional_response', 'acceptable_price', 'target_sport', 'price_clean', 'difficulty_clean', 'strategy_importance_num', 'injury_risk_comfort_num', 'individual_brilliance_num', 'play_season_Spring', 'play_season_Summer', 'play_season_Fall', 'play_season_Winter', 'play_season_None', 'play_location_Outdoor public space', 'play_location_School or university facility', 'play_location_Private sports facility', 'play_location_Community recreation facility', 'play_location_Indoor facility', 'play_location_At home or nearby', 'play_location_Others', 'play_location_None', 'associated_music_Hip-hop or rap', 'associated_music_Pop', 'associated_music_Rock', 'associated_music_Electronic or dance', 'associated_music_No particular type', 'associated_music_Latin', 'associated_music_Country

In [45]:
train_df = train_df.drop(columns=['id'])
val_df = val_df.drop(columns=['id'])
test_df = test_df.drop(columns=['id'])

# Training set and Features

In [46]:
exclude_cols = {
        "id", "difficulty_rating", "strategy_importance", "injury_risk_comfort",
        "individual_brilliance", "play_location", "play_season", "associated_music",
        "associated_foods_drinks", "associated_color", "associated_song",
        "emotional_response", "acceptable_price", "target_sport", "target_sport_num",
        "price_clean", "difficulty_clean", "log_price",
        "associated_foods_drinks_clean", "associated_color_clean",
        "associated_song_clean", "emotional_response_clean"
    }

print(f"[Info] Total features before exclusion: {len(train_df.columns)}")
feature_cols = [c for c in train_df.columns if c not in exclude_cols]
print(f"[Info] Total features extracted: {len(feature_cols)}")

classes = ["Soccer", "Baseball", "Basketball"]
target_col = "target_sport_num"

X_train_raw = train_df[feature_cols]
y_train = train_df[target_col]

X_val_raw = val_df[feature_cols]
y_val = val_df[target_col]

X_test_raw = test_df[feature_cols]
y_test = test_df[target_col]

[Info] Total features before exclusion: 154
[Info] Total features extracted: 133


# Imputation

In [47]:
from sklearn.impute import SimpleImputer

imputer = SimpleImputer(strategy="median")
X_train = imputer.fit_transform(X_train_raw)
X_val = imputer.transform(X_val_raw)
X_test = imputer.transform(X_test_raw)

# Model 1: Softmax

## Hyperparameter Tuning

In [48]:
from sklearn.linear_model import LogisticRegression
from sklearn.metrics import accuracy_score, classification_report, confusion_matrix, precision_score, recall_score, f1_score

print("--- Tuning Softmax Regression ---")

best_lr = None
best_lr_score = -1.0

c_candidates = [0.01, 0.1, 0.5, 1.0, 5.0, 10.0]

for c in c_candidates:
    lr = LogisticRegression(
        C=c,
        max_iter=1000,
        random_state=42
    )

    lr.fit(X_train, y_train)

    train_acc = accuracy_score(y_train, lr.predict(X_train))
    val_acc = accuracy_score(y_val, lr.predict(X_val))

    print(
        f"C={c:5.2f} | "
        f"Train Accuracy: {train_acc:.4f} | "
        f"Validation Accuracy: {val_acc:.4f}"
    )

    if val_acc > best_lr_score:
        best_lr_score = val_acc
        best_lr = lr

print(f"\nBest C: {best_lr.C}")
print(f"Best Validation Accuracy: {best_lr_score:.4f}")

--- Tuning Softmax Regression ---
C= 0.01 | Train Accuracy: 0.8752 | Validation Accuracy: 0.8837
C= 0.10 | Train Accuracy: 0.9097 | Validation Accuracy: 0.9380
C= 0.50 | Train Accuracy: 0.9458 | Validation Accuracy: 0.9457
C= 1.00 | Train Accuracy: 0.9589 | Validation Accuracy: 0.9457
C= 5.00 | Train Accuracy: 0.9787 | Validation Accuracy: 0.9457
C=10.00 | Train Accuracy: 0.9852 | Validation Accuracy: 0.9380

Best C: 0.5
Best Validation Accuracy: 0.9457


In [49]:
train_pred_lr = best_lr.predict(X_train)

accuracy = accuracy_score(y_train, train_pred_lr)
precision = precision_score(y_train, train_pred_lr, average="macro")
recall = recall_score(y_train, train_pred_lr, average="macro")
f1 = f1_score(y_train, train_pred_lr, average="macro")

print("--- Softmax Regression: Training Performance ---")
print(f"Training Accuracy:        {accuracy:.4f}")
print(f"Training Macro Precision: {precision:.4f}")
print(f"Training Macro Recall:    {recall:.4f}")
print(f"Training Macro F1:        {f1:.4f}")

print("\nConfusion Matrix:")
print(confusion_matrix(y_train, train_pred_lr))

print("\nClassification Report:")
print(classification_report(y_train, train_pred_lr))

--- Softmax Regression: Training Performance ---
Training Accuracy:        0.9458
Training Macro Precision: 0.9462
Training Macro Recall:    0.9458
Training Macro F1:        0.9459

Confusion Matrix:
[[189   9   5]
 [  5 193   5]
 [  2   7 194]]

Classification Report:
              precision    recall  f1-score   support

           0       0.96      0.93      0.95       203
           1       0.92      0.95      0.94       203
           2       0.95      0.96      0.95       203

    accuracy                           0.95       609
   macro avg       0.95      0.95      0.95       609
weighted avg       0.95      0.95      0.95       609



In [50]:
val_pred_lr = best_lr.predict(X_val)

accuracy = accuracy_score(y_val, val_pred_lr)
precision = precision_score(y_val, val_pred_lr, average="macro")
recall = recall_score(y_val, val_pred_lr, average="macro")
f1 = f1_score(y_val, val_pred_lr, average="macro")

print("--- Softmax Regression: Validation Performance ---")
print(f"Validation Accuracy:        {accuracy:.4f}")
print(f"Validation Macro Precision: {precision:.4f}")
print(f"Validation Macro Recall:    {recall:.4f}")
print(f"Validation Macro F1:        {f1:.4f}")

print("\nConfusion Matrix:")
print(confusion_matrix(y_val, val_pred_lr))

print("\nClassification Report:")
print(classification_report(y_val, val_pred_lr))

--- Softmax Regression: Validation Performance ---
Validation Accuracy:        0.9457
Validation Macro Precision: 0.9457
Validation Macro Recall:    0.9457
Validation Macro F1:        0.9457

Confusion Matrix:
[[40  1  2]
 [ 2 41  0]
 [ 1  1 41]]

Classification Report:
              precision    recall  f1-score   support

           0       0.93      0.93      0.93        43
           1       0.95      0.95      0.95        43
           2       0.95      0.95      0.95        43

    accuracy                           0.95       129
   macro avg       0.95      0.95      0.95       129
weighted avg       0.95      0.95      0.95       129



In [51]:
# Store results

rf_results = {
    "Model": f"Softmax Regression (C={lr.C})",
    "Train Acc": accuracy_score(y_train, train_pred_lr),
    "Val Acc": accuracy_score(y_val, val_pred_lr),
    "Val Precision": precision_score(y_val, val_pred_lr, average="macro"),
    "Val Recall": recall_score(y_val, val_pred_lr, average="macro"),
    "Val Macro-F1": f1_score(y_val, val_pred_lr, average="macro")
}
results.append(rf_results)

In [52]:
from sklearn.model_selection import RepeatedStratifiedKFold, cross_val_score

cv = RepeatedStratifiedKFold(
    n_splits=5,
    n_repeats=3,
    random_state=42
)

scores = cross_val_score(
    lr,
    X_train_raw,
    y_train,
    cv=cv,
    scoring="accuracy"
)

print("Mean:", scores.mean())
print("Std:", scores.std())
print("Individual scores:", scores)

Mean: 0.8604525132095923
Std: 0.024023342353041467
Individual scores: [0.86065574 0.87704918 0.86065574 0.86065574 0.87603306 0.86885246
 0.85245902 0.8442623  0.83606557 0.90082645 0.79508197 0.89344262
 0.86065574 0.86885246 0.85123967]


# Model 2: Random Forest Only

In [53]:
from sklearn.ensemble import RandomForestClassifier

print("--- Tuning Random Forest ---")

best_rf = None
best_rf_score = -1.0

depth_candidates = [5, 10, 14, 15, 16,  20, None]

for depth in depth_candidates:
    
    rf = RandomForestClassifier(
        max_depth=depth,
        random_state=42
    )

    rf.fit(X_train, y_train)

    train_acc = accuracy_score(y_train, rf.predict(X_train))
    val_acc = accuracy_score(y_val, rf.predict(X_val))

    print(
        f"max_depth={str(depth):4s} | "
        f"Training Accuracy: {train_acc:.4f} | "
        f"Validation Accuracy: {val_acc:.4f}"
    )

    if val_acc > best_rf_score:
        best_rf_score = val_acc
        best_rf = rf

print("\nSelected parameters:")
print(f"Maximum depth: {best_rf.max_depth}")
print(f"Number of trees: {best_rf.n_estimators}")
print(f"Best validation accuracy: {best_rf_score:.4f}")

--- Tuning Random Forest ---
max_depth=5    | Training Accuracy: 0.9179 | Validation Accuracy: 0.9225
max_depth=10   | Training Accuracy: 0.9754 | Validation Accuracy: 0.9380
max_depth=14   | Training Accuracy: 0.9934 | Validation Accuracy: 0.9380
max_depth=15   | Training Accuracy: 0.9951 | Validation Accuracy: 0.9535
max_depth=16   | Training Accuracy: 0.9951 | Validation Accuracy: 0.9225
max_depth=20   | Training Accuracy: 0.9951 | Validation Accuracy: 0.9457
max_depth=None | Training Accuracy: 0.9951 | Validation Accuracy: 0.9380

Selected parameters:
Maximum depth: 15
Number of trees: 100
Best validation accuracy: 0.9535


In [54]:
train_pred_rf = best_rf.predict(X_train)

accuracy = accuracy_score(y_train, train_pred_rf)
precision = precision_score(y_train, train_pred_rf, average="macro")
recall = recall_score(y_train, train_pred_rf, average="macro")
f1 = f1_score(y_train, train_pred_rf, average="macro")

print("--- Random Forest: Training Performance ---")
print(f"Training Accuracy:        {accuracy:.4f}")
print(f"Training Macro Precision: {precision:.4f}")
print(f"Training Macro Recall:    {recall:.4f}")
print(f"Training Macro F1:        {f1:.4f}")

print("\nConfusion Matrix:")
print(confusion_matrix(y_train, train_pred_rf))

print("\nClassification Report:")
print(classification_report(y_train, train_pred_rf))

--- Random Forest: Training Performance ---
Training Accuracy:        0.9951
Training Macro Precision: 0.9951
Training Macro Recall:    0.9951
Training Macro F1:        0.9951

Confusion Matrix:
[[202   1   0]
 [  0 203   0]
 [  0   2 201]]

Classification Report:
              precision    recall  f1-score   support

           0       1.00      1.00      1.00       203
           1       0.99      1.00      0.99       203
           2       1.00      0.99      1.00       203

    accuracy                           1.00       609
   macro avg       1.00      1.00      1.00       609
weighted avg       1.00      1.00      1.00       609



In [55]:
val_pred_rf = best_rf.predict(X_val)

accuracy = accuracy_score(y_val, val_pred_rf)
precision = precision_score(y_val, val_pred_rf, average="macro")
recall = recall_score(y_val, val_pred_rf, average="macro")
f1 = f1_score(y_val, val_pred_rf, average="macro")

print("--- Random Forest: Validation Performance ---")
print(f"Validation Accuracy:        {accuracy:.4f}")
print(f"Validation Macro Precision: {precision:.4f}")
print(f"Validation Macro Recall:    {recall:.4f}")
print(f"Validation Macro F1:        {f1:.4f}")

print("\nConfusion Matrix:")
print(confusion_matrix(y_val, val_pred_rf))

print("\nClassification Report:")
print(classification_report(y_val, val_pred_rf))



--- Random Forest: Validation Performance ---
Validation Accuracy:        0.9535
Validation Macro Precision: 0.9540
Validation Macro Recall:    0.9535
Validation Macro F1:        0.9537

Confusion Matrix:
[[40  3  0]
 [ 2 41  0]
 [ 1  0 42]]

Classification Report:
              precision    recall  f1-score   support

           0       0.93      0.93      0.93        43
           1       0.93      0.95      0.94        43
           2       1.00      0.98      0.99        43

    accuracy                           0.95       129
   macro avg       0.95      0.95      0.95       129
weighted avg       0.95      0.95      0.95       129



In [56]:
# Store results

rf_results = {
    "Model": f"Random Forest (depth={best_rf.max_depth})",
    "Train Acc": accuracy_score(y_train, train_pred_rf),
    "Val Acc": accuracy_score(y_val, val_pred_rf),
    "Val Precision": precision_score(y_val, val_pred_rf, average="macro"),
    "Val Recall": recall_score(y_val, val_pred_rf, average="macro"),
    "Val Macro-F1": f1_score(y_val, val_pred_rf, average="macro")
}
results.append(rf_results)

In [57]:
import pandas as pd

feature_importance = pd.DataFrame({
    "Feature": feature_cols,
    "Importance": best_rf.feature_importances_
})

feature_importance = feature_importance.sort_values(
    by="Importance",
    ascending=False
)

print(feature_importance)

                            Feature  Importance
106      associated_song_tfidf_waka    0.069531
12    play_location_Indoor facility    0.066945
70     associated_color_tfidf_green    0.066590
73    associated_color_tfidf_orange    0.062224
6                play_season_Winter    0.052863
..                              ...         ...
64    associated_color_tfidf_colors    0.000054
78      associated_color_tfidf_team    0.000051
77     associated_color_tfidf_sport    0.000043
74   associated_color_tfidf_picture    0.000000
59      associated_color_tfidf_ball    0.000000

[133 rows x 2 columns]


In [58]:
# shuffle label to check for leakage
import numpy as np

y_train_shuffle = np.random.permutation(y_train)
X_valid = val_df[feature_cols]
Y_valid = val_df[target_col]

rf.fit(X_train_raw, y_train_shuffle)

pred = rf.predict(X_valid)

print(accuracy_score(Y_valid, pred))

0.32558139534883723


In [59]:
# Testing individual features for leakage
from sklearn.ensemble import RandomForestClassifier
from sklearn.metrics import accuracy_score

for feature in feature_cols:
    model = RandomForestClassifier(random_state=42)

    model.fit(X_train_raw[[feature]], y_train)

    predictions = model.predict(X_valid[[feature]])

    print(
        feature,
        accuracy_score(Y_valid, predictions)
    )

strategy_importance_num 0.4186046511627907
injury_risk_comfort_num 0.43410852713178294
individual_brilliance_num 0.46511627906976744
play_season_Spring 0.3953488372093023
play_season_Summer 0.3178294573643411
play_season_Fall 0.3953488372093023
play_season_Winter 0.5038759689922481
play_season_None 0.3333333333333333
play_location_Outdoor public space 0.3488372093023256
play_location_School or university facility 0.3953488372093023
play_location_Private sports facility 0.5038759689922481
play_location_Community recreation facility 0.4806201550387597
play_location_Indoor facility 0.6434108527131783
play_location_At home or nearby 0.5503875968992248
play_location_Others 0.3875968992248062
play_location_None 0.3333333333333333
associated_music_Hip-hop or rap 0.5426356589147286
associated_music_Pop 0.3643410852713178
associated_music_Rock 0.3333333333333333
associated_music_Electronic or dance 0.4186046511627907
associated_music_No particular type 0.3953488372093023
associated_music_Latin 

In [60]:
# Leakage sanity check using cross-validation
from sklearn.model_selection import RepeatedStratifiedKFold, cross_val_score

cv = RepeatedStratifiedKFold(
    n_splits=5,
    n_repeats=3,
    random_state=42
)

scores = cross_val_score(
    rf,
    X_train_raw,
    y_train,
    cv=cv,
    scoring="accuracy"
)

print("Mean:", scores.mean())
print("Std:", scores.std())
print("Individual scores:", scores)

Mean: 0.8878110463803459
Std: 0.02382724263974127
Individual scores: [0.87704918 0.94262295 0.85245902 0.90163934 0.90082645 0.85245902
 0.90163934 0.87704918 0.86885246 0.91735537 0.86885246 0.90163934
 0.87704918 0.90163934 0.87603306]


# Model 3: Naive Bayes Only

In [61]:
from sklearn.naive_bayes import MultinomialNB

print("\n--- Tuning Model 3: Multinomial Naive Bayes ---")

text_cols = [c for c in feature_cols if "_tfidf_" in c]

X_train_nb = train_df[text_cols]
X_val_nb = val_df[text_cols]

alpha_candidates = [0.1, 0.5, 1.0, 2.0]

nb_results = []

for alpha in alpha_candidates:

    nb = MultinomialNB(alpha=alpha)
    nb.fit(X_train_nb, y_train)

    train_pred = nb.predict(X_train_nb)
    val_pred = nb.predict(X_val_nb)

    train_acc = accuracy_score(y_train, train_pred)
    val_acc = accuracy_score(y_val, val_pred)
    val_f1 = f1_score(y_val, val_pred, average="macro")

    nb_results.append({
        "alpha": alpha,
        "model": nb,
        "train_accuracy": train_acc,
        "val_accuracy": val_acc,
        "val_f1": val_f1
    })

    print(
        f"alpha={alpha:3.1f} | "
        f"Training Accuracy: {train_acc:.4f} | "
        f"Validation Accuracy: {val_acc:.4f} | "
        f"Validation Macro-F1: {val_f1:.4f}"
    )

# Select highest validation accuracy, then highest macro-F1.
# If both tie, prefer the smaller positive alpha.
best_nb_result = sorted(
    nb_results,
    key=lambda x: (
        -x["val_accuracy"],
        -x["val_f1"],
        x["alpha"]
    )
)[0]

best_nb = best_nb_result["model"]
best_nb_score = best_nb_result["val_accuracy"]

print("\nSelected parameters:")
print(f"Best alpha: {best_nb.alpha}")
print(f"Best validation accuracy: {best_nb_score:.4f}")
print(f"Best validation Macro-F1: {best_nb_result['val_f1']:.4f}")

# Final predictions using selected model
train_pred_nb = best_nb.predict(X_train_nb)
val_pred_nb = best_nb.predict(X_val_nb)


--- Tuning Model 3: Multinomial Naive Bayes ---
alpha=0.1 | Training Accuracy: 0.9015 | Validation Accuracy: 0.9147 | Validation Macro-F1: 0.9149
alpha=0.5 | Training Accuracy: 0.9015 | Validation Accuracy: 0.9147 | Validation Macro-F1: 0.9149
alpha=1.0 | Training Accuracy: 0.8998 | Validation Accuracy: 0.9070 | Validation Macro-F1: 0.9072
alpha=2.0 | Training Accuracy: 0.8949 | Validation Accuracy: 0.9070 | Validation Macro-F1: 0.9072

Selected parameters:
Best alpha: 0.1
Best validation accuracy: 0.9147
Best validation Macro-F1: 0.9149


In [62]:
print("\n--- Naive Bayes: Training Performance ---")

print("Accuracy:", accuracy_score(y_train, train_pred_nb))
print("Precision:", precision_score(y_train, train_pred_nb, average="macro"))
print("Recall:", recall_score(y_train, train_pred_nb, average="macro"))
print("F1 Score:", f1_score(y_train, train_pred_nb, average="macro"))

print("\nConfusion Matrix:")
print(confusion_matrix(y_train, train_pred_nb))

print("\nClassification Report:")
print(classification_report(y_train, train_pred_nb))


--- Naive Bayes: Training Performance ---
Accuracy: 0.9014778325123153
Precision: 0.9031662818032412
Recall: 0.9014778325123153
F1 Score: 0.9017298495166646

Confusion Matrix:
[[182  15   6]
 [  6 187  10]
 [  7  16 180]]

Classification Report:
              precision    recall  f1-score   support

           0       0.93      0.90      0.91       203
           1       0.86      0.92      0.89       203
           2       0.92      0.89      0.90       203

    accuracy                           0.90       609
   macro avg       0.90      0.90      0.90       609
weighted avg       0.90      0.90      0.90       609



In [63]:
print("\n--- Naive Bayes: Validation Performance ---")

print("Accuracy:", accuracy_score(y_val, val_pred_nb))
print("Precision:", precision_score(y_val, val_pred_nb, average="macro"))
print("Recall:", recall_score(y_val, val_pred_nb, average="macro"))
print("F1 Score:", f1_score(y_val, val_pred_nb, average="macro"))

print("\nConfusion Matrix:")
print(confusion_matrix(y_val, val_pred_nb))

print("\nClassification Report:")
print(classification_report(y_val, val_pred_nb))


--- Naive Bayes: Validation Performance ---
Accuracy: 0.9147286821705426
Precision: 0.9156950484233524
Recall: 0.9147286821705426
F1 Score: 0.9148796939494614

Confusion Matrix:
[[40  2  1]
 [ 3 39  1]
 [ 2  2 39]]

Classification Report:
              precision    recall  f1-score   support

           0       0.89      0.93      0.91        43
           1       0.91      0.91      0.91        43
           2       0.95      0.91      0.93        43

    accuracy                           0.91       129
   macro avg       0.92      0.91      0.91       129
weighted avg       0.92      0.91      0.91       129



In [64]:
nb_results = {
    "Model": f"Multinomial Naive Bayes (alpha={best_nb.alpha})",
    "Train Acc": accuracy_score(y_train, train_pred_nb),
    "Val Acc": accuracy_score(y_val, val_pred_nb),
    "Val Precision": precision_score(y_val, val_pred_nb, average="macro"),
    "Val Recall": recall_score(y_val, val_pred_nb, average="macro"),
    "Val Macro-F1": f1_score(y_val, val_pred_nb, average="macro")
}

results.append(nb_results)

# Model 4: Naive Bayes + Random Forest Ensemble

In [65]:
print("\n--- Tuning Model 4: Random Forest + Naive Bayes Ensemble ---")

weight_candidates = [0.9, 0.8, 0.7, 0.6, 0.5]

best_ensemble_score = -1.0
best_ensemble = None

for rf_weight in weight_candidates:

    nb_weight = 1 - rf_weight

    ensemble_probabilities = (
        rf_weight * best_rf.predict_proba(X_val_raw)
        + nb_weight * best_nb.predict_proba(X_val_nb)
    )

    ensemble_pred = best_rf.classes_[
        ensemble_probabilities.argmax(axis=1)
    ]

    val_acc = accuracy_score(y_val, ensemble_pred)
    val_f1 = f1_score(y_val, ensemble_pred, average="macro")

    print(
        f"RF Weight: {rf_weight:.1f} | "
        f"NB Weight: {nb_weight:.1f} | "
        f"Validation Accuracy: {val_acc:.4f} | "
        f"Macro-F1: {val_f1:.4f}"
    )

    if val_acc > best_ensemble_score:
        best_ensemble_score = val_acc
        best_ensemble = {
            "rf_weight": rf_weight,
            "nb_weight": nb_weight,
            "prediction": ensemble_pred
        }

print("\nSelected Ensemble Weights:")
print(f"Random Forest: {best_ensemble['rf_weight']:.1f}")
print(f"Naive Bayes: {best_ensemble['nb_weight']:.1f}")
print(f"Best Validation Accuracy: {best_ensemble_score:.4f}")


--- Tuning Model 4: Random Forest + Naive Bayes Ensemble ---
RF Weight: 0.9 | NB Weight: 0.1 | Validation Accuracy: 0.9457 | Macro-F1: 0.9460
RF Weight: 0.8 | NB Weight: 0.2 | Validation Accuracy: 0.9380 | Macro-F1: 0.9380
RF Weight: 0.7 | NB Weight: 0.3 | Validation Accuracy: 0.9380 | Macro-F1: 0.9380
RF Weight: 0.6 | NB Weight: 0.4 | Validation Accuracy: 0.9225 | Macro-F1: 0.9226
RF Weight: 0.5 | NB Weight: 0.5 | Validation Accuracy: 0.9147 | Macro-F1: 0.9147

Selected Ensemble Weights:
Random Forest: 0.9
Naive Bayes: 0.1
Best Validation Accuracy: 0.9457


/Library/Frameworks/Python.framework/Versions/3.10/lib/python3.10/site-packages/sklearn/utils/validation.py:2742: UserWarning: X has feature names, but RandomForestClassifier was fitted without feature names
  warnings.warn(
/Library/Frameworks/Python.framework/Versions/3.10/lib/python3.10/site-packages/sklearn/utils/validation.py:2742: UserWarning: X has feature names, but RandomForestClassifier was fitted without feature names
  warnings.warn(
/Library/Frameworks/Python.framework/Versions/3.10/lib/python3.10/site-packages/sklearn/utils/validation.py:2742: UserWarning: X has feature names, but RandomForestClassifier was fitted without feature names
  warnings.warn(
/Library/Frameworks/Python.framework/Versions/3.10/lib/python3.10/site-packages/sklearn/utils/validation.py:2742: UserWarning: X has feature names, but RandomForestClassifier was fitted without feature names
  warnings.warn(
/Library/Frameworks/Python.framework/Versions/3.10/lib/python3.10/site-packages/sklearn/utils/valida

In [66]:
# Get selected weights
rf_weight = best_ensemble["rf_weight"]
nb_weight = best_ensemble["nb_weight"]

# Training probabilities
ensemble_train_probabilities = (
    rf_weight * best_rf.predict_proba(X_train_raw)
    + nb_weight * best_nb.predict_proba(X_train_nb)
)

# Training predictions
ensemble_train_pred = best_rf.classes_[
    ensemble_train_probabilities.argmax(axis=1)
]

# Validation predictions
ensemble_val_pred = best_ensemble["prediction"]

/Library/Frameworks/Python.framework/Versions/3.10/lib/python3.10/site-packages/sklearn/utils/validation.py:2742: UserWarning: X has feature names, but RandomForestClassifier was fitted without feature names
  warnings.warn(


In [67]:
# Ensemble training metrics
print("\n--- Ensemble: Training Performance ---")

print("Accuracy:", accuracy_score(y_train, ensemble_train_pred))
print("Precision:", precision_score(y_train, ensemble_train_pred, average="macro"))
print("Recall:", recall_score(y_train, ensemble_train_pred, average="macro"))
print("F1 Score:", f1_score(y_train, ensemble_train_pred, average="macro"))

print("\nConfusion Matrix:")
print(confusion_matrix(y_train, ensemble_train_pred))

print("\nClassification Report:")
print(classification_report(y_train, ensemble_train_pred))


--- Ensemble: Training Performance ---
Accuracy: 0.993431855500821
Precision: 0.9935587761674718
Recall: 0.993431855500821
F1 Score: 0.9934476374466715

Confusion Matrix:
[[201   2   0]
 [  0 203   0]
 [  0   2 201]]

Classification Report:
              precision    recall  f1-score   support

           0       1.00      0.99      1.00       203
           1       0.98      1.00      0.99       203
           2       1.00      0.99      1.00       203

    accuracy                           0.99       609
   macro avg       0.99      0.99      0.99       609
weighted avg       0.99      0.99      0.99       609



In [68]:
# Ensemble validation metrics
print("\n--- Ensemble: Validation Performance ---")

print("Accuracy:", accuracy_score(y_val, ensemble_val_pred))
print("Precision:", precision_score(y_val, ensemble_val_pred, average="macro"))
print("Recall:", recall_score(y_val, ensemble_val_pred, average="macro"))
print("F1 Score:", f1_score(y_val, ensemble_val_pred, average="macro"))

print("\nConfusion Matrix:")
print(confusion_matrix(y_val, ensemble_val_pred))

print("\nClassification Report:")
print(classification_report(y_val, ensemble_val_pred))


--- Ensemble: Validation Performance ---
Accuracy: 0.9457364341085271
Precision: 0.9464411557434813
Recall: 0.945736434108527
F1 Score: 0.9460026940474132

Confusion Matrix:
[[40  3  0]
 [ 3 40  0]
 [ 1  0 42]]

Classification Report:
              precision    recall  f1-score   support

           0       0.91      0.93      0.92        43
           1       0.93      0.93      0.93        43
           2       1.00      0.98      0.99        43

    accuracy                           0.95       129
   macro avg       0.95      0.95      0.95       129
weighted avg       0.95      0.95      0.95       129



In [69]:
ensemble_results = {
    "Model": f"RF + NB Ensemble ({rf_weight:.0%}/{nb_weight:.0%})",
    "Train Acc": accuracy_score(y_train, ensemble_train_pred),
    "Val Acc": accuracy_score(y_val, ensemble_val_pred),
    "Val Precision": precision_score(y_val, ensemble_val_pred, average="macro"),
    "Val Recall": recall_score(y_val, ensemble_val_pred, average="macro"),
    "Val Macro-F1": f1_score(y_val, ensemble_val_pred, average="macro")
}

results.append(ensemble_results)

# Final Model Comparison & Benchmarks

- Evaluated candidate model families under tuned hyperparameters.
- Results table below summarizes Train/Validation metrics for Report Section 5 (Results).


In [70]:
import pandas as pd

# Convert results into a DataFrame
results_df = pd.DataFrame(results)

# Sort models by validation accuracy
results_df = results_df.sort_values(
    by="Val Acc",
    ascending=False
).reset_index(drop=True)

# Round numerical values
results_df = results_df.round(4)

print("\n--- Model Performance Comparison ---")
display(results_df)


--- Model Performance Comparison ---


,Model,Train Acc,Val Acc,Val Precision,Val Recall,Val Macro-F1
0,Random Forest (depth=15),0.9951,0.9535,0.9540,0.9535,0.9537
1,Softmax Regression (C=10.0),0.9458,0.9457,0.9457,0.9457,0.9457
2,RF + NB Ensemble (90%/10%),0.9934,0.9457,0.9464,0.9457,0.9460
3,Multinomial Naive Bayes (alpha=0.1),0.9015,0.9147,0.9157,0.9147,0.9149
